# Minecraft Audio: Train From Scratch on EnCodec Tokens

This notebook implements the approach you asked for:

1. Build a **deterministic train/val/test split** from `data/processed` with explicit sound-event coverage checks.
2. Tokenize 4s audio clips with **EnCodec 24kHz at 1.5 kbps** (2 codebooks).
3. Train a **small text-conditioned causal transformer** from scratch on discrete tokens.
4. Generate tokens from text, then decode back to waveform with EnCodec.

## Important note on split coverage
Many sound events in this dataset have only 1-2 variants. To keep every event present across train/val/test, this notebook creates **virtual augmented copies** for sparse events (speed perturbation) in val/test.
This avoids exact raw duplicates while preserving event coverage.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --- Project/data path detection (local + Colab friendly) ---
from pathlib import Path

CWD = Path.cwd()

# If you know your exact dataset location, set it here.
# Example (Colab): Path("/content/data_minecapmf/data")
# Example (local): Path("/home/aruzhan/genAI_minecraft/data_minecapmf/data")
DATA_DIR_OVERRIDE = Path("/content/drive/MyDrive/data_minecapmf/data")


def has_processed_audio(p: Path) -> bool:
    return (p / "processed").exists()


def has_metadata(p: Path) -> bool:
    return (p / "processed" / "_captions.json").exists() or (p / "manifest.csv").exists()


def is_data_dir(p: Path) -> bool:
    return has_processed_audio(p) and has_metadata(p)


def resolve_if_exists(p: Path):
    p = p.expanduser()
    return p.resolve() if p.exists() else None


checked = []
DATA_DIR = None

# 1) Manual override first
if DATA_DIR_OVERRIDE is not None:
    candidate = resolve_if_exists(DATA_DIR_OVERRIDE)
    checked.append(str(DATA_DIR_OVERRIDE))
    if candidate is not None and is_data_dir(candidate):
        DATA_DIR = candidate

# 2) Auto-detect common locations
if DATA_DIR is None:
    candidates = [
        CWD / "data",
        CWD / "data_minecapmf" / "data",
        CWD.parent / "data",
        Path("data"),
        Path("data_minecapmf/data"),
        Path("/home/aruzhan/genAI_minecraft/data_minecapmf/data"),
        Path("/content/data_minecapmf/data"),
        Path("/content/drive/MyDrive/data_minecapmf/data"),
        Path("/kaggle/working/data_minecapmf/data"),
    ]

    for cand in candidates:
        checked.append(str(cand))
        cand_resolved = resolve_if_exists(cand)
        if cand_resolved is not None and is_data_dir(cand_resolved):
            DATA_DIR = cand_resolved
            break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find a valid data directory.\n"
        "Expected: <data_dir>/processed plus either _captions.json or manifest.csv.\n"
        "Set DATA_DIR_OVERRIDE to your exact path, e.g. Path('/content/data_minecapmf/data').\n"
        f"Checked candidates: {checked}"
    )

PROJECT_ROOT = DATA_DIR.parent
AUDIO_ROOT = DATA_DIR / "processed"
CAPTIONS_PATH = AUDIO_ROOT / "_captions.json"
MANIFEST_PATH = DATA_DIR / "manifest.csv"
SPLIT_MANIFEST_PATH = DATA_DIR / "manifest_tvt_coverage.csv"
TOKEN_CACHE_DIR = DATA_DIR / "token_cache_encodec_1p5kbps"
RUN_DIR = PROJECT_ROOT / "runs" / "from_scratch_tokens"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("CWD:", CWD)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("AUDIO_ROOT exists:", AUDIO_ROOT.exists())
print("CAPTIONS_PATH exists:", CAPTIONS_PATH.exists(), "->", CAPTIONS_PATH)
print("MANIFEST_PATH exists:", MANIFEST_PATH.exists(), "->", MANIFEST_PATH)


Mounted at /content/drive
CWD: /content
PROJECT_ROOT: /content/drive/MyDrive/data_minecapmf
DATA_DIR: /content/drive/MyDrive/data_minecapmf/data
AUDIO_ROOT exists: True
CAPTIONS_PATH exists: True -> /content/drive/MyDrive/data_minecapmf/data/processed/_captions.json
MANIFEST_PATH exists: True -> /content/drive/MyDrive/data_minecapmf/data/manifest.csv


## Install Dependencies (if needed)

Uncomment and run once in your notebook environment.


In [ ]:
# %pip install -q --upgrade #   numpy pandas tqdm soundfile librosa #   encodec #   transformers sentencepiece

print("Install cell ready. Uncomment if needed.")


In [ ]:
# --- Imports and reproducibility ---
import os
import re
import json
import math
import random
from dataclasses import dataclass
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("device:", DEVICE)
if DEVICE == "cpu":
    print("Warning: CPU only. Training will be slow. Use CUDA for practical runs.")


torch: 2.10.0+cu128
device: cuda


In [ ]:
# --- Load audio-caption records from metadata ---
records = []

if CAPTIONS_PATH.exists():
    captions = json.loads(CAPTIONS_PATH.read_text(encoding="utf-8"))
    for file_name, caption in sorted(captions.items()):
        wav_path = AUDIO_ROOT / file_name
        if wav_path.exists() and file_name.endswith(".wav"):
            records.append({"file_name": file_name, "caption": caption})
    source_used = "_captions.json"

elif MANIFEST_PATH.exists():
    mf = pd.read_csv(MANIFEST_PATH)
    if "file_name" not in mf.columns or "caption" not in mf.columns:
        raise ValueError(
            f"manifest.csv must contain columns ['file_name', 'caption']; got {list(mf.columns)}"
        )
    for _, r in mf.iterrows():
        file_name = str(r["file_name"])
        caption = str(r["caption"])
        wav_path = AUDIO_ROOT / file_name
        if wav_path.exists() and file_name.endswith(".wav"):
            records.append({"file_name": file_name, "caption": caption})
    source_used = "manifest.csv"

else:
    raise FileNotFoundError(
        "Neither metadata source found. Need either processed/_captions.json or data/manifest.csv"
    )

# de-duplicate by file_name (keep first seen)
seen = set()
dedup = []
for r in records:
    if r["file_name"] in seen:
        continue
    seen.add(r["file_name"])
    dedup.append(r)
records = dedup

print("Metadata source:", source_used)
print("Records:", len(records))
if not records:
    raise RuntimeError("No valid records loaded. Check DATA_DIR and processed files.")
print("Example:", records[0])


Metadata source: _captions.json
Records: 325
Example: {'file_name': 'ambient/cave/cave1.wav', 'caption': 'minecraft cave ambience sound effect'}


In [ ]:
# --- Build event keys and categories ---
variant_re = re.compile(r"_(fast|slow|tgap|wgap)$")

def leaf_category(file_name: str) -> str:
    parts = file_name.split("/")
    if len(parts) >= 3 and parts[0] in {"ambient", "mob"}:
        return "/".join(parts[:2])
    return parts[0]

def event_stem(file_name: str) -> str:
    stem = Path(file_name).stem
    return variant_re.sub("", stem)

def event_key(file_name: str) -> str:
    return f"{leaf_category(file_name)}/{event_stem(file_name)}"

for r in records:
    r["leaf_category"] = leaf_category(r["file_name"])
    r["event_key"] = event_key(r["file_name"])

event_counts = Counter(r["event_key"] for r in records)
print("Distinct events:", len(event_counts))
print("Event size distribution:", Counter(event_counts.values()))
print("Events with <3 variants:", sum(v < 3 for v in event_counts.values()))


Distinct events: 120
Event size distribution: Counter({3: 48, 2: 41, 5: 17, 1: 14})
Events with <3 variants: 55


## Step 1: Coverage-Aware Train/Val/Test Split

Default policy in this notebook is **low-data friendly**:
- Put **all original clips** in `train` (so the model sees every available real sample).
- Build `val` and `test` as deterministic **virtual augmented views** per event.

This guarantees:
- every sound event appears in train, val, and test,
- train keeps maximum data (important for 325-clip regime),
- split is fully deterministic and reproducible.

You can switch to a stricter disjoint policy by setting `KEEP_ALL_ORIGINALS_IN_TRAIN = False` in the next cell.


In [ ]:
# --- Build deterministic coverage-aware split manifest ---
SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}
KEEP_ALL_ORIGINALS_IN_TRAIN = True


def add_row(out_rows, rec, split, virtual_aug="none", is_virtual=0, source_file_name=None):
    src = rec["file_name"] if source_file_name is None else source_file_name
    out_rows.append({
        "file_name": rec["file_name"],
        "source_file_name": src,
        "caption": rec["caption"],
        "leaf_category": rec["leaf_category"],
        "event_key": rec["event_key"],
        "split": split,
        "virtual_aug": virtual_aug,
        "is_virtual": int(is_virtual),
    })


def allocate_counts(n, ratios):
    # Enforce at least 1 sample in each split for n>=3.
    n_train = max(1, int(round(n * ratios["train"])))
    n_val = max(1, int(round(n * ratios["val"])))
    n_test = n - n_train - n_val

    if n_test < 1:
        if n_train >= n_val and n_train > 1:
            n_train -= 1
        elif n_val > 1:
            n_val -= 1
        n_test = 1

    while n_train + n_val + n_test < n:
        n_train += 1

    while n_train + n_val + n_test > n:
        if n_train >= n_val and n_train >= n_test and n_train > 1:
            n_train -= 1
        elif n_val >= n_train and n_val >= n_test and n_val > 1:
            n_val -= 1
        elif n_test > 1:
            n_test -= 1
        else:
            break

    return n_train, n_val, n_test


def build_coverage_split(records, seed=42, keep_all_originals_in_train=True):
    rng = random.Random(seed)
    grouped = defaultdict(list)
    for r in records:
        grouped[r["event_key"]].append(r)

    out = []

    if keep_all_originals_in_train:
        # Keep all original data in train for maximum learning signal.
        for rec in sorted(records, key=lambda x: x["file_name"]):
            add_row(out, rec, "train", "none", 0)

        # Add one val and one test row per event via deterministic virtual augmentations.
        for ek in sorted(grouped):
            items = grouped[ek].copy()
            rng.shuffle(items)

            val_src = items[0]
            test_src = items[1] if len(items) > 1 else items[0]

            add_row(out, val_src, "val", "speed_0.95", 1, source_file_name=val_src["file_name"])
            test_aug = "speed_1.05" if test_src["file_name"] != val_src["file_name"] else "speed_1.08"
            add_row(out, test_src, "test", test_aug, 1, source_file_name=test_src["file_name"])

    else:
        # Stricter disjoint allocation where possible.
        for ek in sorted(grouped):
            items = grouped[ek].copy()
            rng.shuffle(items)
            n = len(items)

            if n >= 3:
                n_train, n_val, n_test = allocate_counts(n, SPLIT_RATIOS)
                idx = 0
                for _ in range(n_train):
                    add_row(out, items[idx], "train", "none", 0)
                    idx += 1
                for _ in range(n_val):
                    add_row(out, items[idx], "val", "none", 0)
                    idx += 1
                for _ in range(n_test):
                    add_row(out, items[idx], "test", "none", 0)
                    idx += 1

            elif n == 2:
                add_row(out, items[0], "train", "none", 0)
                add_row(out, items[1], "val", "none", 0)
                add_row(out, items[1], "test", "speed_1.05", 1, source_file_name=items[1]["file_name"])

            else:  # n == 1
                add_row(out, items[0], "train", "none", 0)
                add_row(out, items[0], "val", "speed_0.95", 1, source_file_name=items[0]["file_name"])
                add_row(out, items[0], "test", "speed_1.05", 1, source_file_name=items[0]["file_name"])

    df = pd.DataFrame(out)

    # stable deterministic order
    df = df.sort_values(["event_key", "split", "file_name", "virtual_aug"]).reset_index(drop=True)

    # add sample ids and token cache keys
    def make_cache_relpath(row):
        stem = row["file_name"].replace("/", "__").replace(".wav", "")
        aug = row["virtual_aug"].replace(".", "p")
        return f"{stem}__{aug}.npy"

    df["token_relpath"] = df.apply(make_cache_relpath, axis=1)
    df["sample_id"] = [f"s{i:05d}" for i in range(len(df))]
    return df


split_df = build_coverage_split(records, seed=SEED, keep_all_originals_in_train=KEEP_ALL_ORIGINALS_IN_TRAIN)
split_df.to_csv(SPLIT_MANIFEST_PATH, index=False)
print("Saved:", SPLIT_MANIFEST_PATH)
print("KEEP_ALL_ORIGINALS_IN_TRAIN:", KEEP_ALL_ORIGINALS_IN_TRAIN)
print("Rows:", len(split_df))
print(split_df.head(3))


Saved: /content/drive/MyDrive/data_minecapmf/data/manifest_tvt_coverage.csv
KEEP_ALL_ORIGINALS_IN_TRAIN: True
Rows: 565
                     file_name             source_file_name  \
0       ambient/cave/cave1.wav       ambient/cave/cave1.wav   
1       ambient/cave/cave1.wav       ambient/cave/cave1.wav   
2  ambient/cave/cave1_slow.wav  ambient/cave/cave1_slow.wav   

                                     caption leaf_category  \
0       minecraft cave ambience sound effect  ambient/cave   
1       minecraft cave ambience sound effect  ambient/cave   
2  slow minecraft cave ambience sound effect  ambient/cave   

            event_key  split virtual_aug  is_virtual  \
0  ambient/cave/cave1   test  speed_1.05           1   
1  ambient/cave/cave1  train        none           0   
2  ambient/cave/cave1  train        none           0   

                          token_relpath sample_id  
0  ambient__cave__cave1__speed_1p05.npy    s00000  
1        ambient__cave__cave1__none.npy    s00001

In [ ]:
# --- Split diagnostics ---
print("Rows per split:")
print(split_df["split"].value_counts())

print("\nOriginal vs virtual rows:")
print(split_df["is_virtual"].value_counts())

print("\nOriginal rows per split:")
print(split_df[split_df["is_virtual"] == 0]["split"].value_counts())

# event coverage check
coverage = split_df.groupby("event_key")["split"].apply(lambda x: set(x.tolist()))
missing = {k: sorted({"train", "val", "test"} - v) for k, v in coverage.items() if {"train", "val", "test"} - v}
print("\nEvents missing any split:", len(missing))

# unique underlying files per split
uniq_by_split = split_df.groupby("split")["file_name"].nunique()
print("\nUnique base files per split:")
print(uniq_by_split)

# category distribution
print("\nLeaf category counts by split:")
print(pd.crosstab(split_df["leaf_category"], split_df["split"]))


Rows per split:
split
train    325
test     120
val      120
Name: count, dtype: int64

Original vs virtual rows:
is_virtual
0    325
1    240
Name: count, dtype: int64

Original rows per split:
split
train    325
Name: count, dtype: int64

Events missing any split: 0

Unique base files per split:
split
test     120
train    325
val      120
Name: file_name, dtype: int64

Leaf category counts by split:
split               test  train  val
leaf_category                       
ambient/cave          23     46   23
ambient/underwater     7     14    7
ambient/weather       11     22   11
combat                 8     24    8
damage                 3     11    3
mob/blaze              4     16    4
mob/creeper            2      8    2
mob/endermen           9     35    9
mob/ghast              7     25    7
mob/skeleton           6     20    6
mob/spider             4     12    4
mob/zombie            14     48   14
step                  22     44   22


## Step 2: EnCodec Tokenization at 1.5 kbps

- Model: `encodec_model_24khz`
- Bandwidth: `1.5 kbps`
- Expected codebooks: `2`
- For 4s clips: about `75 frames/s * 4 * 2 = 600` interleaved tokens


In [ ]:
pip install encodec

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 33.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for encodec: filename=encodec-0.1.1-py3-none-any.whl size=45759 sha256=57ba1f136a7782332f75d93e7e950601e9af8facf684735e9d6ffb7835ec86f9
  Stored in directory: /root/.cache/pip/wheels/b8/eb/9f/e13610cc46ab39d3199fbabebd1c3e142d44b679526e0f228a
Successfully built encodec


In [ ]:
# --- Load EnCodec and set tokenization constants ---
from encodec import EncodecModel

ENCODEC_BANDWIDTH = 1.5
DURATION_S = 4.0

encodec_model = EncodecModel.encodec_model_24khz()
encodec_model.set_target_bandwidth(ENCODEC_BANDWIDTH)
encodec_model.to(DEVICE)
encodec_model.eval()

ENCODEC_SR = encodec_model.sample_rate
ENCODEC_CHANNELS = encodec_model.channels
FRAME_RATE = encodec_model.frame_rate

N_CODEBOOKS = 2  # at 1.5 kbps for 24k model
CODEBOOK_SIZE = 1024
TOKEN_SEQ_LEN = int(DURATION_S * FRAME_RATE * N_CODEBOOKS)

print("ENCODEC_SR:", ENCODEC_SR)
print("FRAME_RATE:", FRAME_RATE)
print("N_CODEBOOKS:", N_CODEBOOKS)
print("TOKEN_SEQ_LEN:", TOKEN_SEQ_LEN)
print("CODEBOOK_SIZE:", CODEBOOK_SIZE)


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Downloading: "https://dl.fbaipublicfiles.com/encodec/v0/encodec_24khz-d7cc33bc.th" to /root/.cache/torch/hub/checkpoints/encodec_24khz-d7cc33bc.th


100%|██████████| 88.9M/88.9M [00:03<00:00, 28.6MB/s]


ENCODEC_SR: 24000
FRAME_RATE: 75
N_CODEBOOKS: 2
TOKEN_SEQ_LEN: 600
CODEBOOK_SIZE: 1024


In [ ]:
# --- Audio loading, virtual augmentation, and token helpers ---
def load_audio_mono(path: Path):
    wav, sr = sf.read(str(path), always_2d=False)
    if wav.ndim == 2:
        wav = wav.mean(axis=1)
    wav = wav.astype(np.float32)
    return wav, int(sr)


def apply_virtual_aug(wav: np.ndarray, aug: str):
    if aug == "none":
        return wav
    if aug.startswith("speed_"):
        rate = float(aug.split("_")[1])
        if len(wav) < 16:
            return wav
        return librosa.effects.time_stretch(wav, rate=rate)
    raise ValueError(f"Unknown virtual augmentation: {aug}")


def standardize_audio(wav: np.ndarray, sr: int, target_sr: int, duration_s: float):
    if sr != target_sr:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)

    target_len = int(duration_s * target_sr)
    if len(wav) < target_len:
        wav = np.pad(wav, (0, target_len - len(wav)))
    else:
        wav = wav[:target_len]

    wav = np.clip(wav, -1.0, 1.0)
    return wav.astype(np.float32)


def interleave_codes(codes: torch.Tensor) -> torch.Tensor:
    # codes: [K, T] -> [T*K], order [c1_t1, c2_t1, c1_t2, c2_t2, ...]
    return codes.transpose(0, 1).reshape(-1)


def deinterleave_tokens(tokens: torch.Tensor, n_codebooks: int) -> torch.Tensor:
    # tokens: [T*K] -> [K, T]
    t = tokens.numel() // n_codebooks
    return tokens.view(t, n_codebooks).transpose(0, 1)


def pad_or_trim_1d(x: torch.Tensor, target_len: int, pad_value: int = 0):
    if x.numel() < target_len:
        pad = torch.full((target_len - x.numel(),), pad_value, dtype=x.dtype)
        return torch.cat([x, pad], dim=0)
    return x[:target_len]


def tokenize_row_with_encodec(row):
    wav_path = AUDIO_ROOT / row["file_name"]
    wav, sr = load_audio_mono(wav_path)
    wav = apply_virtual_aug(wav, row["virtual_aug"])
    wav = standardize_audio(wav, sr=sr, target_sr=ENCODEC_SR, duration_s=DURATION_S)

    # EnCodec expects [B, C, T]
    x = torch.from_numpy(wav).to(torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        encoded_frames = encodec_model.encode(x)

    # For 24k model segment=None, we generally have one frame.
    # robustly concatenate in case multiple frames are returned
    codes = torch.cat([ef[0] for ef in encoded_frames], dim=-1)  # [B, K, T]
    codes = codes[0].cpu().to(torch.long)  # [K, T]

    if codes.shape[0] != N_CODEBOOKS:
        raise RuntimeError(f"Expected {N_CODEBOOKS} codebooks at 1.5kbps, got {codes.shape[0]}")

    tokens = interleave_codes(codes)  # [T*K]
    tokens = pad_or_trim_1d(tokens, TOKEN_SEQ_LEN, pad_value=0)
    return tokens


def decode_tokens_with_encodec(tokens_1d: torch.Tensor):
    tokens_1d = tokens_1d.to(torch.long)
    tokens_1d = pad_or_trim_1d(tokens_1d, TOKEN_SEQ_LEN, pad_value=0)
    codes = deinterleave_tokens(tokens_1d, N_CODEBOOKS).unsqueeze(0).to(DEVICE)  # [1, K, T]
    encoded_frames = [(codes, None)]
    with torch.no_grad():
        wav = encodec_model.decode(encoded_frames)
    # wav: [B, C, T]
    return wav[0, 0].detach().cpu().numpy()


In [ ]:
# --- Build token cache for split rows ---
TOKEN_CACHE_DIR.mkdir(parents=True, exist_ok=True)

cache_miss = 0
for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc="Tokenizing rows"):
    out_path = TOKEN_CACHE_DIR / row["token_relpath"]
    if out_path.exists():
        continue
    tokens = tokenize_row_with_encodec(row)
    np.save(out_path, tokens.numpy().astype(np.int16))
    cache_miss += 1

print("Token cache dir:", TOKEN_CACHE_DIR)
print("New token files created:", cache_miss)
print("Total token files:", len(list(TOKEN_CACHE_DIR.glob("*.npy"))))


Tokenizing rows:   0%|          | 0/565 [00:00<?, ?it/s]

Token cache dir: /content/drive/MyDrive/data_minecapmf/data/token_cache_encodec_1p5kbps
New token files created: 565
Total token files: 565


In [ ]:
# --- Quick token/decode sanity check ---
sample_row = split_df.iloc[0]
sample_tokens = np.load(TOKEN_CACHE_DIR / sample_row["token_relpath"])
print("sample file:", sample_row["file_name"])
print("sample split:", sample_row["split"], "aug:", sample_row["virtual_aug"])
print("token shape:", sample_tokens.shape, "min:", sample_tokens.min(), "max:", sample_tokens.max())

recon = decode_tokens_with_encodec(torch.tensor(sample_tokens, dtype=torch.long))
recon_path = RUN_DIR / "sanity_reconstruction.wav"
sf.write(str(recon_path), recon, ENCODEC_SR)
print("Saved reconstruction:", recon_path)


sample file: ambient/cave/cave1.wav
sample split: test aug: speed_1.05
token shape: (600,) min: 0 max: 1022
Saved reconstruction: /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/sanity_reconstruction.wav


## Step 3: Small Text-Conditioned Transformer (from scratch)

Design choices (aligned with your plan):
- Decoder-only causal transformer with cross-attention to text encoder memory.
- Frozen text encoder: `t5-small` encoder.
- Token sequence length: `600` (4s, 1.5kbps, 2 codebooks interleaved).
- Codebook vocab: `1024`, plus BOS token.


In [ ]:
# --- Data pipeline for training ---
from transformers import AutoTokenizer, T5EncoderModel, get_cosine_schedule_with_warmup

BOS_ID = CODEBOOK_SIZE
VOCAB_SIZE = CODEBOOK_SIZE + 1  # only BOS extra token
TEXT_MODEL_NAME = "t5-small"
TEXT_MAX_LEN = 32

# Hyperparameters (from your spec; adjust by hardware)
CFG = {
    "layers": 10,
    "d_model": 512,
    "n_heads": 8,
    "ffn_dim": 2048,
    "dropout": 0.1,
    "lr": 3e-4,
    "betas": (0.9, 0.95),
    "weight_decay": 0.1,
    "batch_size": 16 if DEVICE == "cuda" else 2,
    "epochs": 100,
    "grad_accum_steps": 1,
    "warmup_ratio": 0.05,
    "min_lr": 1e-5,
    "max_grad_norm": 1.0,
    "eval_every_steps": 100,
}

print(CFG)

class TokenCaptionDataset(Dataset):
    def __init__(self, df_split: pd.DataFrame, token_cache_dir: Path):
        self.df = df_split.reset_index(drop=True)
        self.token_cache_dir = token_cache_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        toks = np.load(self.token_cache_dir / row["token_relpath"]).astype(np.int64)
        toks = torch.from_numpy(toks)
        return {
            "tokens": toks,
            "caption": row["caption"],
            "event_key": row["event_key"],
            "file_name": row["file_name"],
        }


def collate_fn(batch):
    tokens = torch.stack([b["tokens"] for b in batch], dim=0)  # [B, L]
    captions = [b["caption"] for b in batch]
    return {
        "tokens": tokens,
        "captions": captions,
    }

train_df = split_df[split_df["split"] == "train"].copy()
val_df = split_df[split_df["split"] == "val"].copy()
test_df = split_df[split_df["split"] == "test"].copy()

train_ds = TokenCaptionDataset(train_df, TOKEN_CACHE_DIR)
val_ds = TokenCaptionDataset(val_df, TOKEN_CACHE_DIR)
test_ds = TokenCaptionDataset(test_df, TOKEN_CACHE_DIR)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=0, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=0, collate_fn=collate_fn)

print("train/val/test rows:", len(train_ds), len(val_ds), len(test_ds))
print("steps/epoch:", math.ceil(len(train_loader) / CFG["grad_accum_steps"]))

text_tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_encoder = T5EncoderModel.from_pretrained(TEXT_MODEL_NAME).to(DEVICE)
text_encoder.eval()
for p in text_encoder.parameters():
    p.requires_grad = False

print("Loaded frozen text encoder:", TEXT_MODEL_NAME)


{'layers': 10, 'd_model': 512, 'n_heads': 8, 'ffn_dim': 2048, 'dropout': 0.1, 'lr': 0.0003, 'betas': (0.9, 0.95), 'weight_decay': 0.1, 'batch_size': 16, 'epochs': 100, 'grad_accum_steps': 1, 'warmup_ratio': 0.05, 'min_lr': 1e-05, 'max_grad_norm': 1.0, 'eval_every_steps': 100}
train/val/test rows: 325 120 120
steps/epoch: 21


Loading weights:   0%|          | 0/51 [00:00<?, ?it/s]

Loaded frozen text encoder: t5-small


In [ ]:
# --- Small causal decoder with cross-attention to text memory ---
class TextCondTokenTransformer(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        n_heads: int,
        num_layers: int,
        ffn_dim: int,
        dropout: float,
        max_seq_len: int,
        n_codebooks: int,
        text_dim: int,
    ):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.n_codebooks = n_codebooks

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.codebook_emb = nn.Embedding(n_codebooks, d_model)

        self.text_proj = nn.Linear(text_dim, d_model)

        layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.decoder = nn.TransformerDecoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids, text_hidden, text_padding_mask=None):
        # input_ids: [B, T], text_hidden: [B, S, text_dim]
        B, T = input_ids.shape
        if T > self.max_seq_len:
            raise ValueError(f"input length {T} > max_seq_len {self.max_seq_len}")

        pos = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, T)
        cb = (torch.arange(T, device=input_ids.device) % self.n_codebooks).unsqueeze(0).expand(B, T)

        x = self.token_emb(input_ids) + self.pos_emb(pos) + self.codebook_emb(cb)
        mem = self.text_proj(text_hidden)

        causal_mask = torch.triu(
            torch.full((T, T), float("-inf"), device=input_ids.device),
            diagonal=1,
        )

        h = self.decoder(
            tgt=x,
            memory=mem,
            tgt_mask=causal_mask,
            memory_key_padding_mask=text_padding_mask,
        )
        h = self.norm(h)
        logits = self.lm_head(h)
        return logits


model = TextCondTokenTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=CFG["d_model"],
    n_heads=CFG["n_heads"],
    num_layers=CFG["layers"],
    ffn_dim=CFG["ffn_dim"],
    dropout=CFG["dropout"],
    max_seq_len=TOKEN_SEQ_LEN,
    n_codebooks=N_CODEBOOKS,
    text_dim=text_encoder.config.d_model,
).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {num_params/1e6:.2f}M")


Model params: 43.66M


In [ ]:
# --- Training utilities ---
def encode_text_batch(captions):
    tok = text_tokenizer(
        captions,
        padding=True,
        truncation=True,
        max_length=TEXT_MAX_LEN,
        return_tensors="pt",
    )
    tok = {k: v.to(DEVICE) for k, v in tok.items()}
    with torch.no_grad():
        out = text_encoder(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"])
    text_hidden = out.last_hidden_state
    text_padding_mask = tok["attention_mask"] == 0  # True means ignore
    return text_hidden, text_padding_mask


def batch_to_inputs_targets(tokens):
    # tokens: [B, L] with values in [0, 1023]
    B, L = tokens.shape
    bos = torch.full((B, 1), BOS_ID, dtype=torch.long, device=tokens.device)
    inp = torch.cat([bos, tokens[:, :-1]], dim=1)
    tgt = tokens
    return inp, tgt


def evaluate_loss(model, loader, max_batches=None):
    model.eval()
    losses = []
    with torch.no_grad():
        for bi, batch in enumerate(loader):
            if max_batches is not None and bi >= max_batches:
                break
            tokens = batch["tokens"].to(DEVICE)
            text_hidden, text_padding_mask = encode_text_batch(batch["captions"])
            inp, tgt = batch_to_inputs_targets(tokens)
            logits = model(inp, text_hidden, text_padding_mask=text_padding_mask)
            loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), tgt.reshape(-1))
            losses.append(loss.item())
    return float(np.mean(losses)) if losses else float("nan")


In [ ]:
# --- Train loop ---
optimizer = AdamW(
    model.parameters(),
    lr=CFG["lr"],
    betas=CFG["betas"],
    weight_decay=CFG["weight_decay"],
)

steps_per_epoch = math.ceil(len(train_loader) / CFG["grad_accum_steps"])
total_steps = CFG["epochs"] * steps_per_epoch
warmup_steps = int(CFG["warmup_ratio"] * total_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print("total_steps:", total_steps, "warmup_steps:", warmup_steps)

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

global_step = 0
best_val = float("inf")
history = []

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    running = []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{CFG['epochs']}")
    optimizer.zero_grad(set_to_none=True)

    for i, batch in enumerate(pbar, start=1):
        tokens = batch["tokens"].to(DEVICE)
        text_hidden, text_padding_mask = encode_text_batch(batch["captions"])
        inp, tgt = batch_to_inputs_targets(tokens)

        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda"), dtype=torch.float16):
            logits = model(inp, text_hidden, text_padding_mask=text_padding_mask)
            loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), tgt.reshape(-1))
            loss = loss / CFG["grad_accum_steps"]

        scaler.scale(loss).backward()

        if i % CFG["grad_accum_steps"] == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CFG["max_grad_norm"])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            step_loss = float(loss.item() * CFG["grad_accum_steps"])
            running.append(step_loss)
            pbar.set_postfix({"loss": f"{np.mean(running[-20:]):.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})

            if global_step % CFG["eval_every_steps"] == 0:
                val_loss = evaluate_loss(model, val_loader, max_batches=None)
                train_loss = float(np.mean(running[-100:])) if running else float("nan")
                history.append({
                    "step": global_step,
                    "epoch": epoch,
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                    "lr": scheduler.get_last_lr()[0],
                })
                print(f"\nstep={global_step} train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

                if val_loss < best_val:
                    best_val = val_loss
                    ckpt = {
                        "model_state": model.state_dict(),
                        "config": CFG,
                        "best_val": best_val,
                        "global_step": global_step,
                        "token_seq_len": TOKEN_SEQ_LEN,
                        "vocab_size": VOCAB_SIZE,
                        "bos_id": BOS_ID,
                        "n_codebooks": N_CODEBOOKS,
                    }
                    torch.save(ckpt, RUN_DIR / "best_model.pt")
                    print("Saved best checkpoint ->", RUN_DIR / "best_model.pt")

# save final artifacts
hist_df = pd.DataFrame(history)
hist_path = RUN_DIR / "train_history.csv"
hist_df.to_csv(hist_path, index=False)
torch.save({"model_state": model.state_dict(), "config": CFG, "global_step": global_step}, RUN_DIR / "last_model.pt")
print("Training complete.")
print("Best val:", best_val)
print("History:", hist_path)
print("Final checkpoint:", RUN_DIR / "last_model.pt")


total_steps: 2100 warmup_steps: 105


/tmp/ipykernel_4763/2288218415.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))


Epoch 1/100:   0%|          | 0/21 [00:00<?, ?it/s]

/tmp/ipykernel_4763/2288218415.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda"), dtype=torch.float16):


Epoch 2/100:   0%|          | 0/21 [00:00<?, ?it/s]

...

Epoch 97/100:   0%|          | 0/21 [00:00<?, ?it/s]

Epoch 98/100:   0%|          | 0/21 [00:00<?, ?it/s]

Epoch 99/100:   0%|          | 0/21 [00:00<?, ?it/s]

Epoch 100/100:   0%|          | 0/21 [00:00<?, ?it/s]


step=2100 train_loss=0.0187 val_loss=7.6559
Training complete.
Best val: 4.214591056108475
History: /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/train_history.csv
Final checkpoint: /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/last_model.pt


## Step 4: Inference (Text -> Tokens -> Audio)

Sampling uses top-k + temperature as requested.


In [ ]:
# --- Sampling and waveform generation ---
def sample_from_logits(logits, temperature=0.9, top_k=250):
    # logits: [B, V]
    if temperature <= 0:
        raise ValueError("temperature must be > 0")

    logits = logits / temperature

    # Only sample actual EnCodec token ids [0..1023], never BOS
    logits = logits[:, :CODEBOOK_SIZE]

    if top_k is not None and top_k > 0:
        k = min(top_k, logits.shape[-1])
        vals, idx = torch.topk(logits, k=k, dim=-1)
        probs = torch.softmax(vals, dim=-1)
        next_local = torch.multinomial(probs, num_samples=1)
        next_token = idx.gather(-1, next_local)
    else:
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

    return next_token


def generate_token_sequence(prompt, max_tokens=TOKEN_SEQ_LEN, temperature=0.9, top_k=250):
    model.eval()

    text_hidden, text_padding_mask = encode_text_batch([prompt])

    seq = torch.tensor([[BOS_ID]], dtype=torch.long, device=DEVICE)  # [1,1]
    for _ in tqdm(range(max_tokens), desc="Generating tokens"):
        inp = seq[:, -TOKEN_SEQ_LEN:]
        logits = model(inp, text_hidden, text_padding_mask=text_padding_mask)
        next_token = sample_from_logits(logits[:, -1, :], temperature=temperature, top_k=top_k)
        seq = torch.cat([seq, next_token], dim=1)

    out_tokens = seq[:, 1:1 + max_tokens].squeeze(0).detach().cpu()
    return out_tokens


def save_generated_audio(prompt, out_wav_path, temperature=0.9, top_k=250):
    tokens = generate_token_sequence(prompt, max_tokens=TOKEN_SEQ_LEN, temperature=temperature, top_k=top_k)
    wav = decode_tokens_with_encodec(tokens)
    sf.write(str(out_wav_path), wav, ENCODEC_SR)
    return out_wav_path


PROMPT = "minecraft zombie groaning in a cave with short footsteps"
out_wav = RUN_DIR / "gen_zombie_cave.wav"
path = save_generated_audio(PROMPT, out_wav, temperature=0.9, top_k=250)
print("Saved generated wav:", path)


Generating tokens:   0%|          | 0/600 [00:00<?, ?it/s]

Saved generated wav: /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/gen_zombie_cave.wav


In [ ]:
prompts = [
    "minecraft zombie groaning in a cave with short footsteps",
    "minecraft endermen teleporting sound effect",
    "minecraft ghast crying sound effect",
    "minecraft creeper hissing and exploding"
]

for i, prompt in enumerate(prompts):
    out_wav_name = f"gen_sound_{i+1}.wav"
    out_wav_path = RUN_DIR / out_wav_name
    path = save_generated_audio(prompt, out_wav_path, temperature=0.9, top_k=250)
    print(f"Saved generated wav: {path} for prompt: '{prompt}'")

Generating tokens:   0%|          | 0/600 [00:00<?, ?it/s]

Saved generated wav: /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/gen_sound_1.wav for prompt: 'minecraft zombie groaning in a cave with short footsteps'


Generating tokens:   0%|          | 0/600 [00:00<?, ?it/s]

Saved generated wav: /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/gen_sound_2.wav for prompt: 'minecraft endermen teleporting sound effect'


Generating tokens:   0%|          | 0/600 [00:00<?, ?it/s]

Saved generated wav: /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/gen_sound_3.wav for prompt: 'minecraft ghast crying sound effect'


Generating tokens:   0%|          | 0/600 [00:00<?, ?it/s]

Saved generated wav: /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/gen_sound_4.wav for prompt: 'minecraft creeper hissing and exploding'


In [ ]:
from IPython.display import Audio, display

for i, prompt in enumerate(prompts):
    out_wav_name = f"gen_sound_{i+1}.wav"
    generated_audio_path = RUN_DIR / out_wav_name

    print(f"\n--- Prompt: '{prompt}' ---")
    print(f"Generated Audio: {generated_audio_path}")

    # Load and display generated audio
    try:
        y_gen, sr_gen = librosa.load(generated_audio_path, sr=None)
        display(Audio(y_gen, rate=sr_gen))
    except Exception as e:
        print(f"Error loading generated audio {generated_audio_path}: {e}")


--- Prompt: 'minecraft spider walking sound' ---
Generated Audio: /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/gen_sound_1.wav



--- Prompt: 'minecraft skeleton shooting arrow sound' ---
Generated Audio: /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/gen_sound_2.wav



--- Prompt: 'minecraft blaze breathing fire sound' ---
Generated Audio: /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/gen_sound_3.wav



--- Prompt: 'minecraft ghast crying sound' ---
Generated Audio: /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/gen_sound_4.wav



--- Prompt: 'minecraft zombie groaning sound' ---
Generated Audio: /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/gen_sound_5.wav
Error loading generated audio /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/gen_sound_5.wav: [Errno 2] No such file or directory: '/content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/gen_sound_5.wav'

--- Prompt: 'minecraft creeper hissing sound' ---
Generated Audio: /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/gen_sound_6.wav
Error loading generated audio /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/gen_sound_6.wav: [Errno 2] No such file or directory: '/content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/gen_sound_6.wav'


/tmp/ipykernel_4763/4051392042.py:12: UserWarning: PySoundFile failed. Trying audioread instead.
  y_gen, sr_gen = librosa.load(generated_audio_path, sr=None)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In [ ]:
from IPython.display import Audio, display

# New list of animal-like (Minecraft mob) sounds
prompts = [
    "minecraft spider walking sound",
    "minecraft skeleton shooting arrow sound",
    "minecraft blaze breathing fire sound",
    "minecraft ghast crying sound",
    "minecraft zombie groaning sound",
    "minecraft creeper hissing sound"
]

print("Generating new animal-like sounds...")
generated_audio_paths = []
for i, prompt in enumerate(prompts):
    out_wav_name = f"gen_animal_sound_{i+1}.wav"
    out_wav_path = RUN_DIR / out_wav_name
    path = save_generated_audio(prompt, out_wav_path, temperature=0.9, top_k=250)
    generated_audio_paths.append(path)
    print(f"Saved generated wav: {path} for prompt: '{prompt}'")

print("\nListening to generated and ground truth sounds:")
for i, prompt in enumerate(prompts):
    out_wav_name = f"gen_animal_sound_{i+1}.wav"
    generated_audio_path = RUN_DIR / out_wav_name

    print(f"\n--- Prompt: '{prompt}' ---")
    print(f"Generated Audio: {generated_audio_path}")

    # Load and display generated audio
    try:
        y_gen, sr_gen = librosa.load(generated_audio_path, sr=None)
        display(Audio(y_gen, rate=sr_gen))
    except Exception as e:
        print(f"Error loading generated audio {generated_audio_path}: {e}")

    # Attempt to find a relevant ground truth audio
    keywords = prompt.split()
    gt_row = None
    for keyword in keywords:
        # Simple keyword matching, prioritize specific mob names
        if keyword in ['zombie', 'endermen', 'ghast', 'creeper', 'spider', 'skeleton', 'blaze']:
            gt_candidates = split_df[
                (split_df['caption'].str.contains(keyword, case=False)) &
                (split_df['is_virtual'] == 0)
            ]
            if not gt_candidates.empty:
                # Pick a random one for variety from the original non-virtual dataset
                gt_row = gt_candidates.sample(n=1, random_state=SEED).iloc[0]
                break # Found a good match, move on

    if gt_row is None:
        # Fallback if no specific keyword match is found, try broader terms or just pick a random one
        gt_candidates = split_df[(split_df['is_virtual'] == 0)]
        if not gt_candidates.empty:
            gt_row = gt_candidates.sample(n=1, random_state=SEED).iloc[0]

    if gt_row is not None:
        ground_truth_file_name = gt_row['file_name']
        ground_truth_caption = gt_row['caption']
        ground_truth_audio_path = AUDIO_ROOT / ground_truth_file_name

        print(f"Ground Truth Audio: {ground_truth_audio_path}")
        print(f"Ground Truth Caption: {ground_truth_caption}")

        # Load and display ground truth audio
        try:
            y_gt, sr_gt = librosa.load(ground_truth_audio_path, sr=None)
            display(Audio(y_gt, rate=sr_gt))
        except Exception as e:
            print(f"Error loading ground truth audio {ground_truth_audio_path}: {e}")
    else:
        print("No suitable ground truth audio found for this prompt.")


Generating new animal-like sounds...


Generating tokens:   0%|          | 0/600 [00:00<?, ?it/s]

Saved generated wav: /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/gen_animal_sound_1.wav for prompt: 'minecraft spider walking sound'


Generating tokens:   0%|          | 0/600 [00:00<?, ?it/s]

Saved generated wav: /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/gen_animal_sound_2.wav for prompt: 'minecraft skeleton shooting arrow sound'


...

Ground Truth Audio: /content/drive/MyDrive/data_minecapmf/data/processed/mob/ghast/fireball_seq.wav
Ground Truth Caption: minecraft ghast shooting fireball sound effect



--- Prompt: 'minecraft zombie groaning sound' ---
Generated Audio: /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/gen_animal_sound_5.wav


Ground Truth Audio: /content/drive/MyDrive/data_minecapmf/data/processed/mob/zombie/metal_death_slow.wav
Ground Truth Caption: slow minecraft zombie banging metal and dying sound effect



--- Prompt: 'minecraft creeper hissing sound' ---
Generated Audio: /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/gen_animal_sound_6.wav


Ground Truth Audio: /content/drive/MyDrive/data_minecapmf/data/processed/mob/creeper/say_death_slow.wav
Ground Truth Caption: slow minecraft creeper growling and dying sound effect


In [ ]:
from IPython.display import Audio, display

manual_prompt = input("Enter your prompt to generate audio: ")

print(f"\nGenerating audio for prompt: '{manual_prompt}'")

out_wav_name = "manual_generated_audio.wav"
out_wav_path = RUN_DIR / out_wav_name

try:
    path = save_generated_audio(manual_prompt, out_wav_path, temperature=0.9, top_k=250)
    print(f"Saved generated wav: {path}")

    print(f"\n--- Generated Audio for: '{manual_prompt}' ---")
    try:
        y_gen, sr_gen = librosa.load(generated_audio_path, sr=None)
        display(Audio(y_gen, rate=sr_gen))
    except Exception as e:
        print(f"Error loading generated audio {generated_audio_path}: {e}")

    # Attempt to find a relevant ground truth audio
    keywords = manual_prompt.split()
    gt_row = None
    # Prioritize specific mob names if present in the prompt
    mob_keywords = ['zombie', 'endermen', 'ghast', 'creeper', 'spider', 'skeleton', 'blaze', 'cave', 'water', 'footsteps', 'walking', 'running', 'ambient', 'hit', 'damage', 'fire']

    for keyword in keywords:
        if keyword.lower() in mob_keywords:
            gt_candidates = split_df[
                (split_df['caption'].str.contains(keyword, case=False)) &
                (split_df['is_virtual'] == 0)
            ]
            if not gt_candidates.empty:
                gt_row = gt_candidates.sample(n=1, random_state=SEED).iloc[0]
                break

    if gt_row is None:
        # Fallback to broader search if no specific keyword match
        gt_candidates = split_df[
            (split_df['caption'].str.contains(manual_prompt, case=False, na=False)) &
            (split_df['is_virtual'] == 0)
        ]
        if not gt_candidates.empty:
             gt_row = gt_candidates.sample(n=1, random_state=SEED).iloc[0]

    if gt_row is None:
        # Final fallback: pick a random non-virtual ground truth audio
        gt_candidates = split_df[(split_df['is_virtual'] == 0)]
        if not gt_candidates.empty:
            gt_row = gt_candidates.sample(n=1, random_state=SEED).iloc[0]

    if gt_row is not None:
        ground_truth_file_name = gt_row['file_name']
        ground_truth_caption = gt_row['caption']
        ground_truth_audio_path = AUDIO_ROOT / ground_truth_file_name

        print(f"\n--- Ground Truth Audio (for comparison): {ground_truth_audio_path} ---")
        print(f"Ground Truth Caption: {ground_truth_caption}")

        try:
            y_gt, sr_gt = librosa.load(ground_truth_audio_path, sr=None)
            display(Audio(y_gt, rate=sr_gt))
        except Exception as e:
            print(f"Error loading ground truth audio {ground_truth_audio_path}: {e}")
    else:
        print("No suitable ground truth audio found for comparison.")

except Exception as e:
    print(f"An error occurred during audio generation: {e}")


Enter your prompt to generate audio: minecraft sound: walking on stone

Generating audio for prompt: 'minecraft sound: walking on stone'


Generating tokens:   0%|          | 0/600 [00:00<?, ?it/s]

Saved generated wav: /content/drive/MyDrive/data_minecapmf/runs/from_scratch_tokens/manual_generated_audio.wav

--- Generated Audio for: 'minecraft sound: walking on stone' ---



--- Ground Truth Audio (for comparison): /content/drive/MyDrive/data_minecapmf/data/processed/mob/skeleton/step_walk.wav ---
Ground Truth Caption: minecraft skeleton walking footsteps sound effect
